In [ ]:
from pynq import Overlay, interrupt
from pynq.lib import axigpio
from IPython.display import clear_output

import asyncio
import time

# LOad overlay
overlay = Overlay('MainDesign.bit')
overlay?

In [ ]:
gpio_ip = overlay.axi_gpio_0
gpio_ip.setdirection("in")
gpio_interrupt = gpio_ip.ip2intc_irpt

# Access the register map
gpio_register_map = gpio_ip.register_map
print(gpio_ip.register_map)


In [ ]:
import time
interrupt1 = interrupt.Interrupt('axi_gpio_0/ip2intc_irpt')
if __name__ == '__main__':
    # Main asyncio loop
    loop = asyncio.get_event_loop()
    rotateClkWiseCnt = 0
    rotateAntiClkWiseCnt = 0
    totalRotations = 0
    
    # Enable the GPIO interrupt and clear any existing interrupt
    gpio_register_map.IP_IER.Channel_1_Interrupt_Enable = 1
    gpio_register_map.IP_ISR.Channel_1_Interrupt_Status = 1
    t3 = time.perf_counter()
    while True:
        try:
            #Waiting for the interrupt generated by the A signal
            await interrupt1.wait() 
            
            #Immediately checks if B signal is 0 or 1
            if gpio_register_map.GPIO_DATA.Channel_1_GPIO_DATA == 1 and gpio_register_map.GPIO2_DATA.Channel_2_GPIO_DATA == 0:
                print(rotateClkWiseCnt)
                rotateClkWiseCnt += 1
            if gpio_register_map.GPIO_DATA.Channel_1_GPIO_DATA == 1 and gpio_register_map.GPIO2_DATA.Channel_2_GPIO_DATA == 1:
                print(rotateClkWiseCnt)
                rotateAntiClkWiseCnt += 1    
                
            #Then clears the interrupt signal to be ready for next interrupt
            gpio_register_map.IP_ISR.Channel_1_Interrupt_Status = 1
            
            if rotateClkWiseCnt == 7:
                rotateClkWiseCnt = 0
                totalRotations += 1
            
            if totalRotations == 1:
                print(f"Finished, Print: {rotateClkWiseCnt}")
                t4 = time.perf_counter()
                print(f"Time taken: {t4 - t3}")
                break
            # Run the interrupt handler coroutine until
            #task1 = asyncio.create_task(GPIO_handler())
            #await task1
        except KeyboardInterrupt:
            print("Script interrupted by user.")
            break

In [ ]:
"""
Notes: 
- This still uses asyncio's asynchronous functionalities but not the full architecture.
- The timing matches with the timing observed by oscilliscope but doesn't make much sense:
    + 1 rotation in real life looks to be ~1 second while 1 full rotation takes around 10ms
"""